In [16]:
import pandas as pd
import numpy as np
import wmfdata as wmf

from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType

In [19]:
# Query to get all page_ids classified as medicine-related.
query_isaac = """
SELECT pageid as page_id, wiki_db
FROM isaacj.article_topics_outlinks
WHERE topic = 'STEM.Medicine_&_Health'
and snapshot = '2022-09'
"""

isaac_df = wmf.spark.run(query_isaac)

In [20]:
# Reformat wiki_db column to match project column in pageviews.hourly
isaac_df['wiki_db'] = isaac_df['wiki_db'].str.replace('wiki', '.wikipedia')

# Create a Spark session
spark = SparkSession.builder \
    .appName("Page View Analysis") \
    .getOrCreate()


# Convert the Pandas DataFrame to a Spark DataFrame
isaac_spark_df = spark.createDataFrame(isaac_df)

isaac_spark_df.createOrReplaceTempView("isaac_data")

# View df
isaac_df

,page_id,wiki_db
0,12545,en.wikipedia
1,16535,en.wikipedia
2,46262,en.wikipedia
3,47444,en.wikipedia
4,56285,en.wikipedia
...,...,...
941003,55,pwn.wikipedia
941004,5010,pi.wikipedia
941005,5026,pi.wikipedia
941006,1683,din.wikipedia


In [ ]:
# Query pageviews data

query = """
SELECT id.wiki_db, pvh.page_id, SUM(pvh.view_count) AS total_view_count
FROM wmf.pageview_hourly pvh
INNER JOIN isaac_data id ON pvh.page_id = id.page_id AND pvh.project = id.wiki_db
WHERE pvh.agent_type = 'user'
AND pvh.year = 2022
GROUP BY id.wiki_db, pvh.page_id
"""


# Execute the query to aggregate views
views_df = spark.sql(query)

# Convert Spark DataFrame to Pandas DataFrame
views_df = views_df.toPandas()

# Save page views as .csv for backup
views_df.to_csv("raw_data_medicine_article_count.csv", index = False)

In [24]:
# Aggregate by project and sum total views and count unique page IDs
views_pd = views_df.copy()

aggregated_data = views_pd.groupby('wiki_db').agg(
    total_views=('total_view_count', 'sum'),
    unique_pages=('page_id', 'nunique')
).reset_index()

# Sort the results by total_views in descending order
aggregated_data = aggregated_data.sort_values(by='total_views', ascending=False)

# Save the aggregated data to a CSV file.  Average views (=total views / unique pages) and rank were created in the spreadsheet
aggregated_data.to_csv('aggregated_project_views_summary.csv', index=False)

# Display the aggregated DataFrame
aggregated_data


,wiki_db,total_views,unique_pages
65,en.wikipedia,2656408331,127050
219,ru.wikipedia,564824112,30018
67,es.wikipedia,511307820,32290
56,de.wikipedia,442025973,58034
76,fr.wikipedia,272215549,38225
...,...,...,...
38,bug.wikipedia,457,1
72,ff.wikipedia,79,1
268,tum.wikipedia,40,1
244,srn.wikipedia,38,1
